# 🎯 Customer Retention & LTV Intelligence

End-to-end pipeline: raw data ingestion, cleaning, RFM-based churn risk scoring, and export to SQL for cohort retention and lifetime value analysis.

**Data source:** UCI Machine Learning Repository — Online Retail II dataset (Dec 2009 – Dec 2011, two-year span, used specifically to enable multi-month cohort tracking).

## 1. Data Ingestion
Downloading the raw dataset directly from its public source (skipped automatically if already present locally).

In [1]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np

# 1. Project working directory (generic, machine-independent path)
project_dir = "./Customer_Retention_Project"
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

dataset_url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
zip_path = "online_retail_ii.zip"

# 2. Download only if not already present
if not os.path.exists(zip_path):
    print("Downloading dataset...")
    urllib.request.urlretrieve(dataset_url, zip_path)
    print("Download complete.")
else:
    print("Zip file already exists - skipping download.")

# 3. Extract only if not already extracted
file_exists = any(f.endswith('.xlsx') or f.endswith('.csv') for f in os.listdir('.'))
if not file_exists:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete.")
else:
    print("Data file already extracted - skipping.")

# 4. Load the data
file_name = [f for f in os.listdir('.') if f.endswith('.xlsx') or f.endswith('.csv')][0]
print(f"Reading file: {file_name}")

if file_name.endswith('.xlsx'):
    df = pd.read_excel(file_name)
else:
    df = pd.read_csv(file_name, encoding='ISO-8859-1')

print("Total raw rows:", len(df))
df.head()

Zip file already exists - skipping download.
Data file already extracted - skipping.
Reading file: online_retail_II.xlsx
Total raw rows: 525461


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 2. Data Cleaning & RFM / Churn Risk Scoring
Removing cancelled orders and invalid rows, engineering `TotalAmount`, then computing Recency, Frequency, and Monetary value per customer to assign a churn risk category.

**Churn risk logic (threshold-based, Monetary-aware):**
- **High Value at Risk:** hasn't purchased in over 180 days AND has spent $2,000 or more historically — flagged separately because losing these customers has an outsized revenue impact despite being a small group.
- **High Risk:** hasn't purchased in over 180 days (below the $2,000 threshold).
- **Medium Risk:** hasn't purchased in 90–180 days.
- **Low Risk / Active:** purchased within the last 90 days.

The Monetary condition is checked first, so high-spending dormant customers are never miscategorized as ordinary "High Risk."

In [2]:
# 1. Data cleaning and filtering
df_clean = df.copy()

df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

df_clean = df_clean.dropna(subset=['Customer ID'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['Price']

print("Total rows after cleaning:", len(df_clean))
print("Unique customers:", df_clean['Customer ID'].nunique())

# 2. Build RFM metrics
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'Invoice': 'nunique',                                     # Frequency
    'TotalAmount': 'sum'                                      # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 3. Churn risk classification (Monetary-aware, high-value condition checked first)
def calculate_churn_risk(row):
    if row['Recency'] > 180 and row['Monetary'] >= 2000:
        return 'High Value at Risk'
    elif row['Recency'] > 180:
        return 'High Risk'
    elif 90 < row['Recency'] <= 180:
        return 'Medium Risk'
    else:
        return 'Low Risk / Active'

rfm['ChurnRisk'] = rfm.apply(calculate_churn_risk, axis=1)

print("\nCustomer distribution by churn risk:")
print(rfm['ChurnRisk'].value_counts())

Total rows after cleaning: 407664
Unique customers: 4312

Customer distribution by churn risk:
ChurnRisk
Low Risk / Active     2877
High Risk              796
Medium Risk            610
High Value at Risk      29
Name: count, dtype: int64


## 3. Loading into SQL
Exporting both the cleaned transaction table and the customer RFM/churn table into a SQLite database, using a relative path so the notebook runs on any machine without modification.

In [3]:
import sqlite3

# Relative path - works on any machine, no hardcoded local paths
db_path = "customer_analytics.db"
conn = sqlite3.connect(db_path)

df_clean.to_sql('online_retail_clean', conn, if_exists='replace', index=False)
rfm.to_sql('customer_rfm_scores', conn, if_exists='replace', index=False)

print("Tables exported successfully to SQLite.")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

conn.close()

Tables exported successfully to SQLite.
Tables in database: [('online_retail_clean',), ('customer_rfm_scores',)]


## 4. SQL Analytics: Cohort Retention & Customer Lifetime Value
Running the two core analyses directly from this notebook to verify results before building the Power BI dashboard: a monthly cohort retention matrix, and lifetime value per churn risk segment (including the new High Value at Risk segment).

In [4]:
conn = sqlite3.connect("customer_analytics.db")

# --- Query 1: Cohort Retention Matrix ---
cohort_query = """
WITH First_Purchase AS (
    SELECT
        `Customer ID` AS CustomerID,
        DATE(MIN(InvoiceDate), 'start of month') AS CohortMonth
    FROM online_retail_clean
    GROUP BY `Customer ID`
),
Customer_Activity AS (
    SELECT
        r.`Customer ID` AS CustomerID,
        DATE(r.InvoiceDate, 'start of month') AS ActivityMonth,
        fp.CohortMonth,
        (CAST(STRFTIME('%Y', r.InvoiceDate) AS INTEGER) - CAST(STRFTIME('%Y', fp.CohortMonth) AS INTEGER)) * 12 +
        (CAST(STRFTIME('%m', r.InvoiceDate) AS INTEGER) - CAST(STRFTIME('%m', fp.CohortMonth) AS INTEGER)) AS MonthIndex
    FROM online_retail_clean r
    JOIN First_Purchase fp ON r.`Customer ID` = fp.CustomerID
)
SELECT
    CohortMonth,
    COUNT(DISTINCT CustomerID) AS Total_Cohort_Customers,
    COUNT(DISTINCT CASE WHEN MonthIndex = 0 THEN CustomerID END) AS Month_0,
    COUNT(DISTINCT CASE WHEN MonthIndex = 1 THEN CustomerID END) AS Month_1,
    COUNT(DISTINCT CASE WHEN MonthIndex = 2 THEN CustomerID END) AS Month_2,
    COUNT(DISTINCT CASE WHEN MonthIndex = 3 THEN CustomerID END) AS Month_3,
    COUNT(DISTINCT CASE WHEN MonthIndex = 6 THEN CustomerID END) AS Month_6,
    COUNT(DISTINCT CASE WHEN MonthIndex = 12 THEN CustomerID END) AS Month_12
FROM Customer_Activity
GROUP BY CohortMonth
ORDER BY CohortMonth;
"""

cohort_df = pd.read_sql_query(cohort_query, conn)
print("Cohort Retention Matrix (raw customer counts by month index):")
display(cohort_df)

# --- Query 2: Customer Lifetime Value (CLV) & Risk Segment Metrics ---
clv_query = """
WITH Customer_Value AS (
    SELECT
        r.CustomerID,
        r.ChurnRisk,
        r.Recency,
        r.Frequency,
        r.Monetary,
        ROUND(r.Monetary / r.Frequency, 2) AS AvgOrderValue,
        CAST(JULIANDAY(MAX(t.InvoiceDate)) - JULIANDAY(MIN(t.InvoiceDate)) AS INTEGER) AS CustomerLifespanDays
    FROM customer_rfm_scores r
    JOIN online_retail_clean t ON r.CustomerID = t.`Customer ID`
    GROUP BY r.CustomerID
)
SELECT
    ChurnRisk,
    COUNT(CustomerID) AS TotalCustomers,
    ROUND(SUM(Monetary), 2) AS SegmentTotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgLifetimeValue_CLV,
    ROUND(AVG(AvgOrderValue), 2) AS Overall_AvgOrderValue,
    ROUND(AVG(Frequency), 1) AS AvgPurchaseFrequency,
    ROUND(AVG(CustomerLifespanDays), 0) AS AvgLifespanDays
FROM Customer_Value
GROUP BY ChurnRisk
ORDER BY SegmentTotalRevenue DESC;
"""

clv_df = pd.read_sql_query(clv_query, conn)
print("\nCustomer Lifetime Value by Churn Risk Segment:")
display(clv_df)

# Save the verified queries to a standalone .sql file for the repository
with open("02_SQL_Cohort_Retention_Analysis.sql", "w", encoding="utf-8") as f:
    f.write("-- ============================================================\n")
    f.write("-- Project: Enterprise Customer Retention & LTV Intelligence\n")
    f.write("-- File: 02_SQL_Cohort_Retention_Analysis.sql\n")
    f.write("-- Description: Advanced SQL Analytics using CTEs, Window Logic & Aggregations\n")
    f.write("-- ============================================================\n\n")
    f.write("-- 1. Cohort Retention Matrix (Monthly Retention Tracking)\n")
    f.write(cohort_query.strip() + "\n\n\n")
    f.write("-- 2. Customer Lifetime Value (CLV) & Risk Segment Metrics\n")
    f.write(clv_query.strip() + "\n")

conn.close()
print("\n02_SQL_Cohort_Retention_Analysis.sql saved successfully.")

Cohort Retention Matrix (raw customer counts by month index):


,CohortMonth,Total_Cohort_Customers,Month_0,Month_1,Month_2,Month_3,Month_6,Month_12
0,2009-12-01,955,955,337,319,406,360,237
1,2010-01-01,383,383,79,119,117,99,0
2,2010-02-01,374,374,89,84,109,72,0
3,2010-03-01,443,443,84,102,107,109,0
4,2010-04-01,294,294,57,57,48,81,0
5,2010-05-01,254,254,40,43,44,54,0
6,2010-06-01,270,270,47,51,55,18,0
7,2010-07-01,186,186,29,34,55,0,0
8,2010-08-01,162,162,33,48,52,0,0
9,2010-09-01,243,243,55,57,24,0,0



Customer Lifetime Value by Churn Risk Segment:


,ChurnRisk,TotalCustomers,SegmentTotalRevenue,AvgLifetimeValue_CLV,Overall_AvgOrderValue,AvgPurchaseFrequency,AvgLifespanDays
0,Low Risk / Active,2877,7747009.32,2692.74,393.65,5.6,176.0
1,Medium Risk,610,578712.32,948.71,332.63,2.7,88.0
2,High Risk,796,313907.38,394.36,278.44,1.5,21.0
3,High Value at Risk,29,192374.25,6633.59,2560.50,4.8,72.0



02_SQL_Cohort_Retention_Analysis.sql saved successfully.
